# MKWii RL Training Monitor
Run the cell below to start training and monitor episode rewards in real time.

In [1]:
import subprocess, sys, os

RUNS_DIR = os.path.join(os.path.dirname(os.getcwd()), "runs")
tb_proc = subprocess.Popen(
    [sys.executable, "-m", "tensorboard.main", "--logdir", RUNS_DIR, "--port", "6006"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f"TensorBoard running at http://localhost:6006  (logdir: {RUNS_DIR})")
print("Run the cell below to start training. Stop this cell to shut down TensorBoard.")

TensorBoard running at http://localhost:6006  (logdir: c:\Users\Nicolas\LocalDocuments\HSLU\DSPRO2\runs)
Run the cell below to start training. Stop this cell to shut down TensorBoard.


In [2]:
import subprocess, sys, re, os
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import display
import ipywidgets as widgets
import json

import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(f'You are training on your {torch.cuda.get_device_name(0)}.')
else:
    print('No GPU detected. You are training on a CPU. Training will be very slow.')

matplotlib.rcParams['figure.figsize'] = (12, 5)

PROJECT_ROOT = os.path.dirname(os.getcwd())
START_SCRIPT = os.path.join(PROJECT_ROOT, "scripts", "StartTraining.py")
STATE_FILE = os.path.join(PROJECT_ROOT, "scripts", "training_state.json")

try:
    with open(STATE_FILE) as f:
        episode_offset = json.load(f).get("episode_count", 0)
except Exception:
    episode_offset = 0

# Storage for episode data
p1_rewards = []
p2_rewards = []
p1_episodes = []
p2_episodes = []
episode_count = 0

PATTERN = re.compile(r'\[TrainingProcess\] P(\d) episode \d+ end\. stuck=(\w+) total_reward=([\-\d\.]+)')

plot_output = widgets.Output()
display(plot_output)

def update_plot():
    with plot_output:
        plot_output.clear_output(wait=True)
        fig, ax1 = plt.subplots(1, 1)

        if p1_rewards:
            ax1.plot(p1_episodes, p1_rewards, 'o-', color='#00E5FF', label='P1', linewidth=1.5, markersize=4)
        if p2_rewards:
            ax1.plot(p2_episodes, p2_rewards, 'o-', color='#FF6B6B', label='P2', linewidth=1.5, markersize=4)
        ax1.axhline(y=0, color='white', linestyle='--', alpha=0.3)
        ax1.set_title('Episode Total Reward', color='white')
        ax1.set_xlabel('Episode', color='white')
        ax1.set_ylabel('Total Reward', color='white')
        ax1.legend()
        ax1.set_facecolor('#1a1a2e')
        fig.patch.set_facecolor('#0f0f23')
        ax1.tick_params(colors='white')
        ax1.spines['bottom'].set_color('white')
        ax1.spines['left'].set_color('white')
        ax1.spines['top'].set_visible(False)
        ax1.spines['right'].set_visible(False)

        plt.tight_layout()
        plt.show()

def parse_line(line):
    global episode_count
    line = line.strip()
    if not line:
        return
    print(line, flush=True)
    matches = PATTERN.findall(line)
    for m in matches:
        player = int(m[0])
        reward = float(m[2])
        if player == 1:
            p1_rewards.append(reward)
        else:
            p2_rewards.append(reward)

    new_count = min(len(p1_rewards), len(p2_rewards))
    if new_count > episode_count:
        for i in range(episode_count + 1, new_count + 1):
            p1_episodes.append(episode_offset + i)
            p2_episodes.append(episode_offset + i)
        episode_count = new_count
        update_plot()

print(f"Starting training from: {START_SCRIPT}")
proc = subprocess.Popen(
    [sys.executable, "-u", START_SCRIPT],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in proc.stdout:
        parse_line(line)
except KeyboardInterrupt:
    proc.terminate()
    print("Training stopped.")

proc.wait()
print("Training process exited.")

True
You are training on your NVIDIA GeForce RTX 3060.


Output()

Starting training from: c:\Users\Nicolas\LocalDocuments\HSLU\DSPRO2\scripts\StartTraining.py
[StartTraining] Emulation speed set to 100%.
[StartTraining] Starting run #1 (crashes so far: 0)
[StartTraining] Launching TrainingProcess...
[StartTraining] Waiting for TrainingProcess to be ready...
[TrainingProcess] Starting up...
[TrainingProcess] Ports bound, ready file written.
[TrainingProcess] Waiting for Dolphin...
[StartTraining] TrainingProcess ready.
[StartTraining] Launching Dolphin...
[StartTraining] All systems go.
[TrainingProcess] P1 connected.
[TrainingProcess] P2 connected.
[TrainingProcess] Dolphin window not ready, retry 1/20...
[TrainingProcess] Dolphin window not ready, retry 1/20...
[DolphinCapture] Player 1 ready.[DolphinCapture] Player 2 ready.
[TrainingProcess] P2 episode 1 end. stuck=True total_reward=13.06
[TrainingProcess] P1 episode 1 end. stuck=True total_reward=19.24


[TrainingProcess] P2 episode 2 end. stuck=True total_reward=23.25
[TrainingProcess] P1 episode 2 end. stuck=True total_reward=29.23


[TrainingProcess] P2 episode 3 end. stuck=True total_reward=13.14
[TrainingProcess] P1 episode 3 end. stuck=True total_reward=18.51


...

[TrainingProcess] P2 episode 302 end. stuck=True total_reward=39.78
[TrainingProcess] P1 episode 302 end. stuck=True total_reward=46.17


[TrainingProcess] P2 episode 303 end. stuck=True total_reward=17.75
[TrainingProcess] P1 episode 303 end. stuck=True total_reward=15.72


[TrainingProcess] P2 episode 304 end. stuck=True total_reward=19.20
[TrainingProcess] P1 episode 304 end. stuck=True total_reward=21.20


[TrainingProcess] P2 episode 305 end. stuck=True total_reward=50.42
[TrainingProcess] P1 episode 305 end. stuck=True total_reward=55.89


[StartTraining] Stop requested. Shutting down...
[StartTraining] Training stopped cleanly after 2 episodes, with 1 crashes.
Training process exited.
